# Шаг 2. Предобработка данных (Preprocessing) и инженерия признаков

**Цель модуля:** Подготовить собранную витрину данных (ABT) к подаче в алгоритмы машинного обучения. Реализовать конвейер обработки пропусков, кодирования категориальных переменных, масштабирования числовых признаков и разделения выборки на обучающую и тестовую.

---

## 📥 Шаг 1. Инициализация окружения и загрузка данных

Импортируем ключевые инструменты из библиотеки `scikit-learn` для трансформации данных и настроим рабочее окружение.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import warnings

warnings.filterwarnings('ignore')

# Настройка путей к данным в соответствии с контуром семинаров
PROCESSED_DIR = os.path.join(".", "data", "processed")

abt_csv_path = os.path.abspath(os.path.join(PROCESSED_DIR, "abt_result.csv"))

print(f"Контур путей Preprocessing настроен.\nВитрина данных (ABT): {abt_csv_path}")


---

## 🛠 ЗАДАНИЕ 1: Обработка пропущенных значений (Imputation)
**Бизнес-контекст:** Большинство классических алгоритмов машинного обучения математически не способны обрабатывать пустоты (`NaN`) в матрице объектов-признаков. Прежде чем обучать модель, аналитик должен заполнить пропуски безопасными статистическими оценками (медианой для чисел, модой для категорий), чтобы не потерять ценные клиентские профили при удалении строк.

**Инструкция (TODO):**
1. Загрузите сырую витрину данных из `abt_result.csv`.
2. Разделите колонки на числовые и категориальные.
3. Заполните пропуски в числовых фичах медианным значением, а в категориальных — самым часто встречающимся (модой).

*🤖 Помощь AI-тьютора:*
* `#PREPROC_TASK1_START` — использование класса `SimpleImputer` из модуля `sklearn.impute`.
* `#PREPROC_TASK1_BUG` — будьте внимательны: метод `.fit_transform()` возвращает `numpy`-массив, его нужно обратно обернуть в DataFrame.


In [ ]:
# [MASTER SOLUTION]
if os.path.exists(abt_csv_path):
    df_abt = pd.read_csv(abt_csv_path)
    
    # Отделяем числовые и категориальные колонки (исключая таргет)
    target_col = [c for c in ['Churn', 'Target_Flag'] if c in df_abt.columns][0]
    features_df = df_abt.drop(columns=[target_col])
    
    num_cols = features_df.select_dtypes(include=[np.number]).columns
    cat_cols = features_df.select_dtypes(exclude=[np.number]).columns
    
    # 1. Импутация числовых признаков (Медиана)
    imputer_num = SimpleImputer(strategy='median')
    df_abt[num_cols] = imputer_num.fit_transform(df_abt[num_cols])
    
    # 2. Импутация категориальных признаков (Мода)
    if len(cat_cols) > 0:
        imputer_cat = SimpleImputer(strategy='most_frequent')
        df_abt[cat_cols] = imputer_cat.fit_transform(df_abt[cat_cols])
        
    print(f"✅ Пропуски успешно устранены. Текущий размер датасета: {df_abt.shape}")
else:
    print("⚠️ Файл витрины данных не найден! Запустите пайплайн ETL/Сборки ABT.")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Считайте abt_result.csv в df_abt
# TODO: 1.2. Определите списки num_cols и cat_cols
# TODO: 1.3. Примените SimpleImputer(strategy='median') для чисел и 'most_frequent' для категорий
raise NotImplementedError("Задание 1 не выполнено! Реализуйте логику заполнения пропусков.")

df_abt = ...
# Ваш код импутации...


---

## 🛠 ЗАДАНИЕ 2: Кодирование категориальных признаков (OHE / Label Encoding)
**Бизнес-контекст:** Линейные модели и нейронные сети требуют строго числового векторного входа. Текстовые бизнес-категории (например, "Тип подписки: Premium") необходимо бинаризовать, чтобы алгоритм смог рассчитать веса для каждого состояния признака.

**Инструкция (TODO):**
1. Примените метод One-Hot Encoding к категориальным переменным.
2. Чтобы избежать мультиколлинеарности ("дамми-ловушки"), обязательно удалите первый базисный столбец (параметр `drop_first=True`).

*🤖 Помощь AI-тьютора:*
* `#PREPROC_TASK2_START` — самый быстрый способ реализовать OHE в pandas: метод `pd.get_dummies()`.


In [ ]:
# [MASTER SOLUTION]
if 'df_abt' in locals() or 'df_abt' in globals():
    if len(cat_cols) > 0:
        # Применяем pd.get_dummies к категориальным колонкам
        df_encoded = pd.get_dummies(df_abt, columns=cat_cols, drop_first=True)
        print(f"✅ One-Hot Encoding завершен. Количество колонок выросло с {df_abt.shape[1]} до {df_encoded.shape[1]}")
    else:
        df_encoded = df_abt.copy()
        print("ℹ️ В датасете нет категориальных признаков для кодирования.")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Передайте категориальные колонки в функцию pd.get_dummies
# TODO: 2.2. Установите аргумент drop_first=True
raise NotImplementedError("Задание 2 не выполнено! Реализуйте кодирование категорий.")

# Ваш код OHE...


---

## 🛠 ЗАДАНИЕ 3: Масштабирование признаков (Feature Scaling)
**Бизнес-контекст:** Финансовые показатели измеряются тысячами (LTV, Monetary), а временные — десятками (Tenure, Recency). Без стандартизации дистанционные алгоритмы (KNN) и методы с градиентным спуском будут придавать необоснованно высокий вес признакам с большим абсолютным масштабом.

**Инструкция (TODO):**
1. Исключите целевую переменную из списка масштабируемых.
2. Инициализируйте `StandardScaler`.
3. Обучите трансформер на признаках и примените преобразование (Z-оценка: среднее = 0, дисперсия = 1).

*🤖 Помощь AI-тьютора:*
* `#PREPROC_TASK3_START` — использование `.fit_transform()` с последующей сборкой `pd.DataFrame`.


In [ ]:
# [MASTER SOLUTION]
if 'df_encoded' in locals():
    # Отделяем матрицу признаков X от вектора ответов y
    X_raw = df_encoded.drop(columns=[target_col])
    y = df_encoded[target_col]
    
    # Инициализация скалера
    scaler = StandardScaler()
    
    # Трансформация данных (возвращает numpy ndarray)
    X_scaled_array = scaler.fit_transform(X_raw)
    
    # Возврат к структуре DataFrame для сохранения имен колонок
    df_scaled = pd.DataFrame(X_scaled_array, columns=X_raw.columns)
    
    print("✅ StandardScaler успешно применен ко всем независимым признакам.")
    display(df_scaled.head(3))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Отделите таргет от независимых переменных
# TODO: 3.2. Примените StandardScaler к матрице признаков
raise NotImplementedError("Задание 3 не выполнено! Реализуйте генератор масштабирования.")

# Ваш код стандартизации...


---

## 🛠 ЗАДАНИЕ 4: Разделение выборки на Train и Test
**Бизнес-контекст:** Чтобы объективно оценить реальную (обобщающую) способность ИИ-модели, мы обязаны утаить от нее часть исторических данных (Тестовый набор) во время обучения. На Семинаре 5 вы узнали о важности стратификации.

**Инструкция (TODO):**
1. Воспользуйтесь функцией `train_test_split`.
2. Выделите 25% данных под тестовую выборку (`test_size=0.25`).
3. Установите зерно генератора случайных чисел `random_state=42` для воспроизводимости.
4. Включите стратификацию по целевой переменной (`stratify=y`), чтобы баланс классов оттока сохранился одинаковым и в трейне, и в тесте.

*🤖 Помощь AI-тьютора:*
* `#PREPROC_TASK4_START` — синтаксис `X_train, X_test, y_train, y_test = train_test_split(...)`.


In [ ]:
# [MASTER SOLUTION]
if 'df_scaled' in locals():
    X_train, X_test, y_train, y_test = train_test_split(
        df_scaled, 
        y, 
        test_size=0.25, 
        random_state=42, 
        stratify=y
    )
    
    print(f"📊 Распределение классов в Train:\n{y_train.value_counts(normalize=True)}")
    print(f"📊 Распределение классов в Test:\n{y_test.value_counts(normalize=True)}")
    print(f"✅ Данные разделены. Размер трейна: {X_train.shape}, Размер теста: {X_test.shape}")
    
    # Экспорт подготовленных тензоров для модуля ML
    X_train.to_csv(os.path.join(PROCESSED_DIR, "X_train.csv"), index=False)
    X_test.to_csv(os.path.join(PROCESSED_DIR, "X_test.csv"), index=False)
    y_train.to_csv(os.path.join(PROCESSED_DIR, "y_train.csv"), index=False)
    y_test.to_csv(os.path.join(PROCESSED_DIR, "y_test.csv"), index=False)
    print("💾 Артефакты машинного обучения сохранены в директорию /data/processed/")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 4.1. Вызовите train_test_split с параметрами test_size=0.25, stratify=y, random_state=42
# TODO: 4.2. Сохраните 4 полученных объекта (X_train, X_test, y_train, y_test) в папку processed
raise NotImplementedError("Задание 4 не выполнено! Напишите код разделения выборки.")

# Ваш код Train-Test Split...


---

## 🏗 ФИНАЛЬНЫЙ АУДИТ КАЧЕСТВА ПАЙПЛАЙНА ПРЕДОБРАБОТКИ
Запустите проверочный блок. Он подтвердит, что конвейер генерации признаков успешно создал все необходимые артефакты для следующего этапа — обучения моделей.


In [ ]:
def verify_preprocessing_pipeline():
    print("🚀 Запуск валидации конвейера Data Preprocessing...\n" + "-"*55)
    
    required_files = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]
    missing_files = []
    
    for file in required_files:
        if not os.path.exists(os.path.join(PROCESSED_DIR, file)):
            missing_files.append(file)
            
    if len(missing_files) == 0:
        print("✅ Все тензоры данных успешно сгенерированы и физически присутствуют.")
        print("🎉 ПОЗДРАВЛЯЕМ! Данные готовы к подаче в ML-модели (Baseline_Models.ipynb).")
    else:
        print(f"❌ Ошибка: Отсутствуют следующие файлы: {missing_files}.")
        print("Вернитесь к Заданию 4 и убедитесь, что экспорт CSV прошел успешно.")
    print("-" * 55)

verify_preprocessing_pipeline()